In [1]:
import pandas as pd
import os

SRC  = os.path.join('..','data', 'waves', 'daily_peak_hs_conditions.csv')
OUT  = os.path.join('..','data', 'waves', 'quinta_waves_1990_2026.csv')
FILL = False        # True = fill missing days by linear interpolation (the model needs every day)

src = pd.read_csv(SRC, parse_dates=['peak_datetime'])
waves = pd.DataFrame({
    'date':    src['peak_datetime'].dt.normalize(),
    'Hs_max':  src['Hs_offshore (m)'],
    'Tp':      src['Tp (s)'],
    'MeanDir': src['Dir (deg)'],
}).set_index('date').sort_index()

full = pd.date_range(waves.index.min(), waves.index.max(), freq='D')
missing = full.difference(waves.index)
print(f'{len(waves)} days, {waves.index[0]:%Y-%m-%d} to {waves.index[-1]:%Y-%m-%d}, '
      f'{len(missing)} missing days')

if FILL:
    waves = waves.reindex(full).interpolate(method='time')   # directions stay within 170-240, no wrap-around
    waves.index.name = 'date'
    print(f'filled {len(missing)} days by linear interpolation')

waves.to_csv(OUT, date_format='%Y-%m-%d')

12531 days, 1990-01-03 to 2026-08-27, 855 missing days


In [3]:
"""Future wave time series by monthly resampling of the hindcast (Davidson et al., 2010;
as in Christiaanse et al., 2026): for every future month, draw a random month of the same
calendar month from the hindcast record and copy its daily values."""
import numpy as np
import pandas as pd

SRC    = os.path.join('..','data', 'waves', 'quinta_waves_1990_2026.csv')     # hindcast forcing (date, Hs_max, Tp, MeanDir)
OUT    = os.path.join('..','data', 'waves', 'quinta_waves_2027_2100.csv')
COLS   = {'Hs_max': 'Hs', 'Tp': 'Tp', 'MeanDir': 'Dir'}   # source -> model column names
START, END = '2026-08-01', '2100-12-31'
POOL   = ('1990-01-01', '2026-07-31')      # months to draw from
MIN_DAYS = 20                              # skip source months with fewer days than this
SEED   = 42

# optional wave-climate change (COWCLIP): % change in Hs and Tp, degrees in Dir, ramped
# linearly from 0 in the first forecast year to the full value in RAMP_TO, then held
CLIMATE = {'Hs': 2.1, 'Tp': 1.2, 'Dir': 0.8}            # e.g. {'Hs': 3.4, 'Tp': 3.0, 'Dir': 0.0}
RAMP_TO = 2090

rng = np.random.default_rng(SEED)

src = pd.read_csv(SRC, parse_dates=[0], index_col=0).rename(columns=COLS)[list(COLS.values())]
src = src.loc[POOL[0]:POOL[1]]

# pool of source months, keyed by calendar month
pool = {m: [] for m in range(1, 13)}
for (y, m), g in src.groupby([src.index.year, src.index.month]):
    if len(g) >= MIN_DAYS:
        g = g.copy()
        g.index = g.index.day                      # index by day of month
        pool[m].append((y, g))
print('source months available per calendar month:', {m: len(v) for m, v in pool.items()})

# draw one source month per future month
out, picks = [], []
for start in pd.date_range(START, END, freq='MS'):
    days = pd.date_range(start, start + pd.offsets.MonthEnd(0), freq='D')
    y_src, g = pool[start.month][rng.integers(len(pool[start.month]))]
    block = g.reindex(days.day)                    # missing days stay NaN, as in the hindcast
    block.index = days
    out.append(block)
    picks.append({'target': start.strftime('%Y-%m'), 'source_year': y_src,
                  'n_days': int(block.notna().all(axis=1).sum())})

fut = pd.concat(out)

if CLIMATE:                                        # optional climate-change ramp
    yrs = fut.index.year.to_numpy()
    f = np.clip((yrs - yrs.min()) / (RAMP_TO - yrs.min()), 0, 1)
    fut['Hs']  *= 1 + CLIMATE.get('Hs', 0) / 100 * f
    fut['Tp']  *= 1 + CLIMATE.get('Tp', 0) / 100 * f
    fut['Dir'] += CLIMATE.get('Dir', 0) * f

fut.index.name = 'Time'
fut.to_csv(OUT, date_format='%Y-%m-%d', na_rep='')
pd.DataFrame(picks).to_csv(OUT.replace('.csv', '_source_months.csv'), index=False)

print(f'{len(fut)} days, {fut.index[0]:%Y-%m-%d} to {fut.index[-1]:%Y-%m-%d}, '
      f'{int(fut.Hs.isna().sum())} days without data')
cmp = pd.DataFrame({'hindcast': src.groupby(src.index.month).mean().Hs,
                    'forecast': fut.groupby(fut.index.month).mean().Hs}).round(2)
print('\nmean Hs by month:\n', cmp.to_string())
print('\noverall  hindcast %.3f m | forecast %.3f m' % (src.Hs.mean(), fut.Hs.mean()))

source months available per calendar month: {1: 33, 2: 36, 3: 37, 4: 37, 5: 37, 6: 37, 7: 37, 8: 35, 9: 36, 10: 36, 11: 36, 12: 36}
27181 days, 2026-08-01 to 2100-12-31, 1581 days without data

mean Hs by month:
     hindcast  forecast
1       1.59      1.60
2       1.63      1.63
3       1.78      1.79
4       1.90      1.89
5       1.98      1.98
6       1.99      2.01
7       1.97      2.00
8       1.94      1.96
9       1.92      1.93
10      1.84      1.85
11      1.72      1.77
12      1.61      1.60

overall  hindcast 1.828 m | forecast 1.838 m


In [9]:
"""Make the hindcast and forecast wave files concatenate cleanly for CoSMoS-COAST.

The model does t_w = [hindcast; forecast] and then interp1, which requires a
sorted, unique time vector. This drops any forecast rows that overlap the
hindcast, writes both files with ISO dates, and checks the result.
"""
import pandas as pd
import os

HIND = os.path.join('..','data','cosmos','quintaall_waves_1990_2026.csv')
FORE = os.path.join('..','data','cosmos','quintaall_waves_2026_2100.csv')
COLS = ['Hs', 'Tp', 'Dir']

def load(path):
    d = pd.read_csv(path)
    d.index = pd.to_datetime(d[d.columns[0]], dayfirst=True, format='mixed')
    return d[COLS].sort_index()

hind, fore = load(HIND), load(FORE)
print(f'hindcast {hind.index[0]:%Y-%m-%d} to {hind.index[-1]:%Y-%m-%d}  ({len(hind)} rows)')
print(f'forecast {fore.index[0]:%Y-%m-%d} to {fore.index[-1]:%Y-%m-%d}  ({len(fore)} rows)')

# keep only forecast days after the hindcast ends
cut = fore.index > hind.index[-1]
print(f'dropping {(~cut).sum()} forecast rows on or before {hind.index[-1]:%Y-%m-%d}')
fore = fore[cut]

for df, path in [(hind, HIND), (fore, FORE)]:
    df.index.name = 'Time'
    df.to_csv(path, date_format='%Y-%m-%d')

# check what the model will see
t = hind.index.append(fore.index)
print(f'\ncombined {len(t)} rows, {t[0]:%Y-%m-%d} to {t[-1]:%Y-%m-%d}')
print('strictly increasing:', bool(t.is_monotonic_increasing and not t.duplicated().any()))
gap = (fore.index[0] - hind.index[-1]).days
print(f'gap at the join: {gap} day(s)')

hindcast 1990-01-03 to 2026-08-27  (12531 rows)
forecast 2026-01-08 to 2100-12-31  (27181 rows)
dropping 55 forecast rows on or before 2026-08-27

combined 39657 rows, 1990-01-03 to 2100-12-31
strictly increasing: True
gap at the join: 1 day(s)
